# 04 — GenAI: call-centre notes → structured reasons (Part 4)
Gemini Flash in JSON mode with a schema, validated with Pydantic. The **first 10 notes only** (free tier).

**Key safety:** the API key is typed with `input()` at run time. It never appears in a code cell. Before downloading, clear the key cell's output and use *Edit → Find* to search the notebook for the first characters of your key.

Call budget: 1 (Task 4.1) + 9 (Task 4.2: N004 is blank, no call) + 2 (Task 4.3 V1; V2 reused from 4.2) = **12 calls**, well under 25.

In [ ]:
!pip install -q google-genai pydantic

In [ ]:
import json, time
import pandas as pd
from pydantic import BaseModel, Field, ValidationError
from google import genai
from google.genai import types

MODEL_NAME = "gemini-3-flash-preview"   # Flash model used in Session 22.1
ALLOWED = ["forgot_or_no_reminder", "distance_or_transport", "cost_or_fee",
           "illness_or_emergency", "work_or_schedule", "other"]
API_CALLS = 0   # every real request to Gemini adds 1
LAST_RAW = ""    # text of the most recent reply

In [ ]:
# The key is typed at run time and never stored in the notebook. Clear this cell's output afterwards.
API_KEY = input("Paste your key: ")
client = genai.Client(api_key=API_KEY)
print("Client ready.")

## Task 4.1 — One validated call

In [ ]:
SCHEMA = {
    "type": "object",
    "properties": {"root_cause": {"type": "string"}, "confidence": {"type": "number"},
                   "summary": {"type": "string"}, "recommended_action": {"type": "string"}},
    "required": ["root_cause", "confidence", "summary", "recommended_action"]}

SYSTEM_TEXT = ("You are a careful analyst for an outpatient clinic chain. You read short call-centre notes "
               "and return only the JSON object requested. You never invent facts that are not in the note.")

def build_prompt(note_text):
    """CRAFT prompt: Context, Role, Action, Tone, Format."""
    return f"""CONTEXT: CareLoop Clinics runs eight outpatient clinics in India. After a patient misses an appointment,
the call centre phones them and writes a short note. Notes may be in English or Hinglish (Hindi in Latin script).

ROLE: You are a patient-experience analyst who turns these notes into structured reasons for management.

ACTION: Read the note and decide the single main reason the patient missed the visit.
Choose root_cause from EXACTLY one of these values:
- forgot_or_no_reminder : forgot the visit and/or did not get a reminder SMS/call
- distance_or_transport : clinic too far, no bus/auto, travel problems
- cost_or_fee           : fee too high, cannot afford it this time
- illness_or_emergency  : the patient or a family member was ill, or another emergency
- work_or_schedule      : job, shift, meeting or other schedule clash
- other                 : anything else, INCLUDING when the note does not say why the patient missed
                          (for example the patient could not be reached). If you are unsure, answer other; do not guess.
Also give confidence (0 to 1), a summary of at most 25 words, and one practical recommended_action for the clinic.

TONE: Neutral, factual and concise. Do not blame the patient.

FORMAT: Return only a JSON object with the keys root_cause, confidence, summary, recommended_action.
root_cause must be one of the six values above, written exactly as shown.

NOTE: {note_text}"""

def call_gemini(prompt):
    """Send one request; return the reply text exactly as received."""
    global API_CALLS
    API_CALLS += 1
    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_TEXT, temperature=0, max_output_tokens=1000,
            response_mime_type="application/json", response_schema=SCHEMA,
            thinking_config=types.ThinkingConfig(thinking_level="minimal")))  # delete this line if your model rejects it
    return response.text

def analyse_note(note_text, return_raw=False):
    """Classify one note with the CRAFT prompt. Returns the parsed dict (and the raw text if asked)."""
    global LAST_RAW
    raw = call_gemini(build_prompt(note_text))
    LAST_RAW = raw                     # kept even if json.loads below fails
    data = json.loads(raw)
    return (data, raw) if return_raw else data

In [ ]:
notes = pd.read_csv("data/followup_notes.csv")
n001 = notes.loc[notes["note_id"] == "N001", "note_text"].iloc[0]
n001_data, n001_raw = analyse_note(n001, return_raw=True)
print(n001_data)
print("Fields:", sorted(n001_data.keys()))
print("root_cause allowed:", n001_data.get("root_cause") in ALLOWED)
time.sleep(13)

## Task 4.2 — Validate and process the first 10 notes

In [ ]:
class FollowupInsight(BaseModel):
    root_cause: str = Field(min_length=3)
    confidence: float = Field(ge=0, le=1)
    summary: str = Field(min_length=5)
    recommended_action: str = Field(min_length=5)

def check_allowed(insight):
    """Raise ValueError if root_cause is not one of the six allowed labels."""
    if insight.root_cause not in ALLOWED:
        raise ValueError(f"root_cause '{insight.root_cause}' is not in the allowed list")
    return insight

print("Model and allowed-list check defined.")

In [ ]:
FIELDS = ["root_cause", "confidence", "summary", "recommended_action"]
first10 = notes.head(10)
records = []

for i, (_, row) in enumerate(first10.iterrows()):
    note_text = row["note_text"]
    rec = {"note_id": row["note_id"], "appointment_id": row["appointment_id"]}

    # input validation: NaN first (a float has no .strip()), then empty / spaces only
    if pd.isna(note_text) or str(note_text).strip() == "":
        rec.update(root_cause="other", confidence=0.0, summary="No note recorded",
                   recommended_action="Re-call patient", validation_status="skipped_blank", raw_response="")
        records.append(rec)
        print(f"{row['note_id']}: blank note -> skipped, no API call")
        continue

    LAST_RAW = ""
    try:
        if i > 0:
            time.sleep(13)                     # free-tier rate limit between calls
        data = analyse_note(note_text)
        for f in FIELDS:                        # copy whatever the reply contained, unrepaired
            rec[f] = data.get(f) if isinstance(data, dict) else None
        check_allowed(FollowupInsight(**data))
        rec["validation_status"] = "valid"
    except (json.JSONDecodeError, ValidationError, ValueError, TypeError) as e:
        rec["validation_status"] = "invalid"
        print(f"{row['note_id']}: invalid -> {type(e).__name__}: {str(e).splitlines()[0]}")
    except Exception as e:                      # API/network error: keep going
        rec["validation_status"] = "invalid"
        print(f"{row['note_id']}: API error -> {type(e).__name__}: {e}")
    rec["raw_response"] = LAST_RAW          # the reply exactly as returned (JSON text)
    records.append(rec)
    print(f"{row['note_id']}: {rec.get('root_cause')} ({rec['validation_status']})")

analysed = pd.DataFrame(records, columns=["note_id", "appointment_id"] + FIELDS + ["validation_status", "raw_response"])
analysed.to_csv("notes_analysed.csv", index=False)
print("\nSaved notes_analysed.csv:", analysed.shape)
print(analysed[["note_id", "root_cause", "confidence", "validation_status"]].to_string(index=False))
print("API calls so far:", API_CALLS)

In [ ]:
expected = {"N001": "forgot_or_no_reminder", "N002": "distance_or_transport", "N003": "cost_or_fee",
            "N004": "other", "N005": "illness_or_emergency", "N006": "work_or_schedule",
            "N007": "forgot_or_no_reminder", "N008": "distance_or_transport", "N009": "other",
            "N010": "work_or_schedule"}
analysed["expected"] = analysed["note_id"].map(expected)
non_blank = analysed[analysed["validation_status"] != "skipped_blank"]
print("Agreement with the answer key on non-blank notes:",
      int((non_blank["root_cause"] == non_blank["expected"]).sum()), "of", len(non_blank))
ok = analysed[analysed["validation_status"].isin(["valid", "skipped_blank"])]
print("Confidence within 0-1 on every valid / skipped row:", ok["confidence"].astype(float).between(0, 1).all())
analysed = analysed.drop(columns="expected")

## Task 4.3 — Break it on purpose
### (a) Two deliberately bad replies

In [ ]:
bad_replies = {
    "confidence = 1.7": {"root_cause": "cost_or_fee", "confidence": 1.7,
                         "summary": "Fee felt too high this month.", "recommended_action": "Offer follow-up discount"},
    "missing recommended_action": {"root_cause": "cost_or_fee", "confidence": 0.8,
                                   "summary": "Fee felt too high this month."},
}
for label, reply in bad_replies.items():
    try:
        FollowupInsight(**reply)
        print(label, "-> unexpectedly passed")
    except ValidationError as e:
        print(f"{label} -> caught ValidationError:")
        for err in e.errors():
            print(f"    field={err['loc']}, type={err['type']}, message={err['msg']}")

### (b) Prompt V1 (bare) vs V2 (CRAFT) on N007 (Hinglish) and N009 (ambiguous)
Same schema and settings for both, so the prompt is the only difference. The V2 replies are the ones already obtained in Task 4.2 with the same CRAFT prompt, so they are reused rather than paid for twice.

In [ ]:
def v1_prompt(note_text):
    return f"Classify this note: {note_text}"

compare = []
for nid in ["N007", "N009"]:
    text = notes.loc[notes["note_id"] == nid, "note_text"].iloc[0]
    time.sleep(13)
    raw_v1 = call_gemini(v1_prompt(text))                      # same schema/settings; only the prompt differs
    v1 = json.loads(raw_v1).get("root_cause")
    v2 = analysed.loc[analysed["note_id"] == nid, "root_cause"].iloc[0]   # V2 = CRAFT reply from Task 4.2 (reused)
    compare.append({"note": nid, "prompt": "V1 (bare)", "root_cause": v1, "allowed": v1 in ALLOWED})
    compare.append({"note": nid, "prompt": "V2 (CRAFT)", "root_cause": v2, "allowed": v2 in ALLOWED})

print(pd.DataFrame(compare).to_string(index=False))

**Which prompt to deploy:** V2 (CRAFT). The bare V1 prompt gives the model no list of labels, so nothing guarantees its answer will be one of the six allowed values, and free-text labels cannot be counted across notes. V2 names the six values and says to answer `other` rather than guess. That matters for notes like N009, where the patient could not be reached and no reason is given.

### (c) Call count

In [ ]:
print("Total Gemini API calls in Tasks 4.1-4.3:", API_CALLS)
print("  4.1: 1 (N001) | 4.2: 9 (N004 blank, no call) | 4.3: 2 (V1 only; V2 reused from 4.2)")
assert API_CALLS < 25

## Task 4.4 (a) — Combine the reasons with the booking data

In [ ]:
notes_df = pd.read_csv("notes_analysed.csv")
appts = pd.read_csv("appointments_clean.csv")
merged = notes_df.merge(appts, on="appointment_id", how="left")
print("Merged rows:", len(merged))
print(merged[["note_id", "root_cause", "reminder_sent", "lead_days", "clinic_id"]].to_string(index=False))
print()
print(pd.crosstab(merged["root_cause"], merged["reminder_sent"], margins=True))